# Занятие 7. SQL-аналитика: COUNT, SUM, AVG, MIN, MAX, GROUP BY, HAVING

## Цель занятия

На прошлых занятиях мы научились:
- создавать таблицы;
- добавлять данные;
- фильтровать данные;
- связывать таблицы через `FOREIGN KEY`;
- объединять данные через `JOIN`.

Сегодня изучаем SQL-аналитику.

К концу занятия студент должен уметь:

- считать количество строк через `COUNT`;
- считать сумму через `SUM`;
- считать среднее через `AVG`;
- находить минимум через `MIN`;
- находить максимум через `MAX`;
- группировать данные через `GROUP BY`;
- фильтровать группы через `HAVING`.

---

## Зачем нужна SQL-аналитика

SQL нужен не только для хранения данных.

Он помогает быстро отвечать на вопросы:

- сколько всего заказов;
- какая общая сумма продаж;
- средний чек;
- сколько кандидатов по каждой вакансии;
- сколько тренировок по каждой цели;
- какие категории дают больше всего выручки.

---

## Где это используется в реальной жизни

Интернет-магазин:
- общая выручка;
- топ категорий;
- средний чек.

HR:
- количество кандидатов по статусам;
- средний score кандидатов;
- максимальная ожидаемая зарплата.

Питание:
- средние калории;
- сумма белков;
- продукты по категориям.

Спорт:
- средняя длительность тренировок;
- сумма сожжённых калорий;
- количество тренировок по типам.

---

## Основные агрегатные функции

| Функция | Что делает |
|---|---|
| COUNT | считает строки |
| SUM | считает сумму |
| AVG | считает среднее |
| MIN | находит минимум |
| MAX | находит максимум |

---

## Что такое GROUP BY

`GROUP BY` группирует строки по одному или нескольким столбцам.

Пример:

```sql
SELECT category, SUM(revenue)
FROM sales
GROUP BY category;
```

Означает:
- сгруппировать продажи по категориям;
- посчитать сумму выручки для каждой категории.

---

## Что такое HAVING

`WHERE` фильтрует строки ДО группировки.

`HAVING` фильтрует группы ПОСЛЕ группировки.

Пример:

```sql
SELECT category, SUM(revenue)
FROM sales
GROUP BY category
HAVING SUM(revenue) > 100000;
```

---

## Зачем это нужно в дипломе

SQL-аналитика делает дипломный проект сильнее.

Студент может показать:
- отчёты;
- статистику;
- агрегаты;
- сравнение категорий;
- простую аналитику по данным.

Это особенно полезно для проектов:
- HR;
- питание;
- спорт;
- продажи;
- обучение;
- заявки клиентов.

---

## Связь с GitHub

На этом занятии студент должен сохранить:

- solved-ноутбук;
- TODO-ноутбук;
- SQL-запросы аналитики;
- пример аналитического отчёта для своей темы диплома.

Рекомендуемая ветка:

```bash
git checkout main
git pull origin main
git checkout -b feature/lesson-07-sql-analytics
```

Рекомендуемый commit:

```bash
git add .
git commit -m "feat: add sql analytics queries"
git push -u origin feature/lesson-07-sql-analytics
```

## Ячейка 1. Подключаем sqlite3 и создаём базу

Создаём базу `analytics_lesson07.db`.

В ней будет таблица продаж `sales`.

In [ ]:
import sqlite3

connection = sqlite3.connect("analytics_lesson07.db")
cursor = connection.cursor()

print("База данных подключена")

## Ячейка 2. Создаём таблицу sales

Таблица хранит продажи:

- товар;
- категория;
- город;
- количество;
- цена;
- выручка.

In [ ]:
cursor.execute("""
CREATE TABLE IF NOT EXISTS sales (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    product TEXT,
    category TEXT,
    city TEXT,
    quantity INTEGER,
    price INTEGER,
    revenue INTEGER
)
""")

connection.commit()

print("Таблица sales создана")

## Ячейка 3. Очищаем таблицу и добавляем данные

Перед добавлением очищаем таблицу, чтобы при повторном запуске не было дублей.

In [ ]:
cursor.execute("DELETE FROM sales")

sales = [
    ("Ноутбук", "Техника", "Казань", 2, 70000, 140000),
    ("Мышь", "Техника", "Казань", 10, 1500, 15000),
    ("Клавиатура", "Техника", "Москва", 5, 3500, 17500),
    ("Книга Python", "Книги", "Москва", 8, 2500, 20000),
    ("Книга SQL", "Книги", "Казань", 6, 2200, 13200),
    ("Стол", "Мебель", "Уфа", 3, 12000, 36000),
    ("Кресло", "Мебель", "Уфа", 2, 18000, 36000),
    ("Блокнот", "Канцелярия", "Казань", 30, 300, 9000),
    ("Ручка", "Канцелярия", "Москва", 50, 100, 5000),
    ("Монитор", "Техника", "Уфа", 4, 25000, 100000)
]

cursor.executemany(
    "INSERT INTO sales (product, category, city, quantity, price, revenue) VALUES (?, ?, ?, ?, ?, ?)",
    sales
)

connection.commit()

print("Данные продаж добавлены")

## Ячейка 4. COUNT — количество строк

Посчитаем, сколько всего записей о продажах есть в таблице.

In [ ]:
cursor.execute("SELECT COUNT(*) FROM sales")

total_rows = cursor.fetchone()[0]

print("Количество строк:", total_rows)

assert total_rows == 10

## Ячейка 5. SUM — общая выручка

Посчитаем сумму столбца `revenue`.

In [ ]:
cursor.execute("SELECT SUM(revenue) FROM sales")

total_revenue = cursor.fetchone()[0]

print("Общая выручка:", total_revenue)

assert total_revenue > 0

## Ячейка 6. AVG, MIN, MAX

Посчитаем:

- среднюю цену;
- минимальную цену;
- максимальную цену.

In [ ]:
cursor.execute("SELECT AVG(price), MIN(price), MAX(price) FROM sales")

avg_price, min_price, max_price = cursor.fetchone()

print("Средняя цена:", round(avg_price, 2))
print("Минимальная цена:", min_price)
print("Максимальная цена:", max_price)

assert max_price >= min_price

## Ячейка 7. GROUP BY по категории

Посчитаем выручку по каждой категории.

In [ ]:
cursor.execute("""
SELECT category, SUM(revenue)
FROM sales
GROUP BY category
""")

revenue_by_category = cursor.fetchall()

for row in revenue_by_category:
    print(row)

assert len(revenue_by_category) >= 1

## Ячейка 8. GROUP BY по городу

Посчитаем:

- количество продаж;
- общую выручку;
- среднюю цену

по каждому городу.

In [ ]:
cursor.execute("""
SELECT city, COUNT(*), SUM(revenue), AVG(price)
FROM sales
GROUP BY city
""")

city_report = cursor.fetchall()

for row in city_report:
    print("Город:", row[0], "| Продаж:", row[1], "| Выручка:", row[2], "| Средняя цена:", round(row[3], 2))

assert len(city_report) >= 1

## Ячейка 9. ORDER BY с агрегатами

Отсортируем категории по выручке от большей к меньшей.

In [ ]:
cursor.execute("""
SELECT category, SUM(revenue) AS total_revenue
FROM sales
GROUP BY category
ORDER BY total_revenue DESC
""")

sorted_categories = cursor.fetchall()

for row in sorted_categories:
    print(row)

assert sorted_categories[0][1] >= sorted_categories[-1][1]

## Ячейка 10. HAVING и закрытие соединения

`HAVING` фильтрует группы после `GROUP BY`.

Покажем только категории, где выручка больше 30000.

In [ ]:
cursor.execute("""
SELECT category, SUM(revenue) AS total_revenue
FROM sales
GROUP BY category
HAVING SUM(revenue) > 30000
ORDER BY total_revenue DESC
""")

big_categories = cursor.fetchall()

print("Категории с выручкой больше 30000:")
for row in big_categories:
    print(row)

assert len(big_categories) >= 1

connection.close()
print("Соединение закрыто")

# Итог занятия 7

Сегодня вы научились:

- считать строки через `COUNT`;
- считать сумму через `SUM`;
- считать среднее через `AVG`;
- находить минимум и максимум;
- группировать данные через `GROUP BY`;
- сортировать агрегированные данные;
- фильтровать группы через `HAVING`.

## Что коммитить в GitHub

- solved-ноутбук;
- TODO-ноутбук;
- SQL-запросы аналитики;
- аналитический отчёт под свою тему диплома.

```bash
git add .
git commit -m "feat: add sql analytics queries"
git push -u origin feature/lesson-07-sql-analytics
```